## The Core Rule

Every selection-and-iteration algorithm has the same three-part shape: **initialize a result variable before the loop, decide inside the loop, report after the loop.** The `if` filters which elements change the result; the loop supplies the elements one at a time.

In [ ]:
// CODE_RUNNER: Count how many scores are passing. Change passingMark, then re-run.
public class CountPassing {
    public static void main(String[] args) {
        int[] scores = {72, 58, 91, 64, 45, 88};
        int passingMark = 70;

        int count = 0;                          // 1. INITIALIZE before the loop
        for (int i = 0; i < scores.length; i++) {
            if (scores[i] >= passingMark) {     // 2. DECIDE inside the loop
                count++;
            }
        }
        System.out.println(count + " of " + scores.length + " scores are passing.");  // 3. REPORT after
    }
}

CountPassing.main(null);

### Reference Guide

#### The Four Standard Algorithms

Almost every Unit 2 free-response task is one of these four, or two of them in the same loop.

| Algorithm | Initialize before the loop | Update inside the loop | Report after the loop |
|---|---|---|---|
| Sum or accumulate | `int total = 0;` | `total += data[i];` | the running total |
| Count | `int count = 0;` | `count++;` guarded by an `if` | how many elements matched |
| Minimum or maximum | `int max = data[0];` | `if (data[i] > max) max = data[i];` | the extreme value |
| Any or all flag | `boolean flag = false;` | `flag = true;` guarded by an `if` | the boolean answer |

#### Loop-and-Decide Grammar

| Rule | Why it matters |
|---|---|
| Declare the result variable before the loop | A variable declared inside the loop is rebuilt every pass, so the answer only reflects the final element |
| Put the `if` inside the loop | Selection is what filters which elements are allowed to change the result |
| Never reassign the accumulator to its starting value inside the loop | Resetting discards everything counted so far |
| Seed a minimum or maximum from `data[0]`, never from `0` | Seeding with `0` silently fails on data that is entirely negative or entirely positive |
| Print after the loop ends, not inside it | Printing inside the loop reports partial answers once per element |
| Remember the counter is an index | `data[i]` is the value; `i` is only its position |
| Cast before dividing for an average | `total / n` uses integer division and truncates; `(double) total / n` does not |

-----

## 1. LxD Cycle Process

**Empathize:** My classmates could already write an `if` statement from 2.3 and a `for` loop from 2.8, but the moment the two were combined they declared the running variable *inside* the loop. The program compiled, printed a number, and looked finished, so nobody suspected anything. The number was just the last element instead of the total, and there was no error message to point at.

**Define:**

*   **POV:** AP CSA students need one repeatable shape for loop-and-decide algorithms because they can already write both halves but lose track of which variable has to live outside the loop.
*   **Learning Goal:** Students will implement sum, count, minimum or maximum, and flag algorithms by initializing the result before the loop, deciding inside it, and reporting after it.

**Ideate:**

*   **HMW Question:** How might we make a variable's scope visible enough that students stop declaring accumulators inside the loop?
*   **Activity:** Predict-then-run on a maximum seeded with `0` over all-negative data, so students watch a wrong answer print confidently before they fix the seed.

**Prototype & Test:** I ran this with my project team. They finished the sum-with-a-condition case in under a minute, but every one of them said a maximum seeded with `0` would still work on negative temperatures. That mistake turned out to be the most useful thing in the lesson, so I promoted it from a footnote into the Popcorn Hack (documented in section 5).

-----

## 2. Lesson Plan

**Learning Objective:** By the end of this lesson, you will be able to implement the four standard selection-and-iteration algorithms (sum, count, minimum or maximum, and flag) by placing each variable in the correct scope, and explain why a wrongly seeded maximum fails without producing an error.

**Success Criteria:** Given an array and a rule stated in English, you can write a loop that produces the right answer on the first try, including on data that is entirely negative, and you can say which line would break if the result variable moved inside the loop.

### Tech Talk & Introduction (3 minutes)

You already have both halves of this topic. Unit 2.8 gave you the loop that walks an array, and Unit 2.3 gave you the `if` that makes a decision. Topic 2.9 is only about **where each variable lives**.

Three lines, in this order, every single time:

1.  **Before the loop** — declare and initialize whatever you are building up.
2.  **Inside the loop** — use an `if` to decide whether this element changes it.
3.  **After the loop** — print or return it.

*   ✅ **Do this:** `int total = 0;` above the loop, `total += data[i];` inside it
*   ❌ **Don't do this:** `int total = 0;` *inside* the loop, which resets it on every pass

Why do we do this? Because this bug never announces itself. Java is perfectly happy to declare a fresh variable on every iteration, so the program compiles, runs, and prints a plausible-looking number. On the exam there is no compiler and no output to check against, so the only defense is recognizing the shape.

### Code Examples

#### A. Simple: Accumulate Only What Passes the Test

In [ ]:
public class SumEvens {
    public static void main(String[] args) {
        int[] values = {4, 7, 10, 3, 6};

        int total = 0;                          // lives OUTSIDE the loop
        for (int i = 0; i < values.length; i++) {
            if (values[i] % 2 == 0) {           // the if decides who contributes
                total += values[i];
            }
        }

        System.out.println("Sum of even values: " + total);   // 4 + 10 + 6 = 20
    }
}

SumEvens.main(null);

#### B. Intermediate: Why a Maximum Must Be Seeded From the Data

This is the single most common way a correct-looking loop returns a wrong answer. Both loops below are identical except for one initialization.

In [ ]:
public class WarmestDay {
    public static void main(String[] args) {
        int[] temps = {-4, -11, -7, -2, -9};    // a cold week: every value is negative

        int wrongMax = 0;                       // no temperature is above 0, so this never updates
        int rightMax = temps[0];                // seeded from real data

        for (int i = 0; i < temps.length; i++) {
            if (temps[i] > wrongMax) { wrongMax = temps[i]; }
            if (temps[i] > rightMax) { rightMax = temps[i]; }
        }

        System.out.println("Seeded with 0        -> " + wrongMax);   // 0, a day that never happened
        System.out.println("Seeded with temps[0] -> " + rightMax);   // -2, the real warmest day
    }
}

WarmestDay.main(null);

Notice what makes this dangerous: `0` is not merely a bad guess, it is a value that **does not appear in the array at all**. The loop reports a day that never happened, and nothing crashes.

#### C. Complex: Four Algorithms in a Single Pass

You do not need four loops for four questions. One walk through the array can carry a sum, a count, a maximum with its index, and a flag at the same time.

In [ ]:
public class ClassReport {
    public static void main(String[] args) {
        int[] scores = {88, 95, 79, 100, 67};
        int goal = 90;

        int total = 0;                    // accumulate
        int countAtGoal = 0;              // count
        int best = scores[0];             // maximum, seeded from the data
        int bestIndex = 0;                // where that maximum lives
        boolean allPassed = true;         // flag, starts optimistic

        for (int i = 0; i < scores.length; i++) {
            total += scores[i];
            if (scores[i] >= goal) { countAtGoal++; }
            if (scores[i] > best) { best = scores[i]; bestIndex = i; }
            if (scores[i] < 70) { allPassed = false; }
        }

        double average = (double) total / scores.length;   // cast first, or 429/5 truncates to 85

        System.out.println("Average           : " + average);       // 85.8
        System.out.println("At or above " + goal + "     : " + countAtGoal);   // 2
        System.out.println("Best score        : " + best + " (index " + bestIndex + ")");  // 100 (index 3)
        System.out.println("Everyone passed   : " + allPassed);     // false, because of the 67
    }
}

ClassReport.main(null);

Two details worth copying. The flag starts at `true` and is only ever turned off, which is how you express "all of them." And `average` casts **before** dividing, because `429 / 5` in integer arithmetic is `85`, not `85.8`.

-----

## 3. Hacks & Practice Tasks

### Prepare Your Submission IPYNB

Complete this quick-start flow so you can begin in about 2 minutes.

1.  Create a new notebook in your portfolio homework area: `_notebooks/homework`.
2.  Add one markdown cell at the top with the frontmatter below.
3.  Add one code cell for the Popcorn Hack and one for the Homework Hack. Keep the `CODE_RUNNER` comment as the first line of each code cell.
4.  Run every cell and read the printed output before you submit.

<pre><code>---
layout: post
title: Implementing Selection and Iteration Algorithms HW
categories: [Java]
lesson_language: Java
lesson_topic: Implementing Selection and Iteration Algorithms HW
lesson_part: interactive
lesson_type: lesson
permalink: /csa/unit_02/2_9/homework
author: githubID
---</code></pre>

### Submission Safety Rules (Read First)

> **Important.** To avoid grading errors, follow these rules exactly.
>
> *   Submit working Java, not pseudocode and not a description of the loop.
> *   Keep `// CODE_RUNNER:` as the first line of every submission cell, and keep `ClassName.main(null);` as the last line. Both are required for the code to run on the page.
> *   Every result variable must be declared **before** its loop. A declaration inside the loop is an automatic deduction even if the printed number happens to be right.
> *   Seed minimums and maximums from `data[0]`. Do not seed from `0`.
> *   Your program must print a labeled line for every value the task asks for.
> *   Use only arrays, `if`, loops, and the operators from Units 2.1 through 2.8. No `Math.max`, no sorting, no library shortcuts.

### Popcorn Hack (In-Class)

> **Tip.** 2-minute challenge: predict first, then run, then paste only your corrected line in chat.

**Task:** The loop below is supposed to print the warmest temperature of the week. Every temperature is below zero. Write down what you think it prints *before* you run it, then run it, then fix the one line that is wrong.

**Expected direction:** your fix should change a single initialization so the answer is guaranteed to be a value that actually appears in the array.

In [ ]:
// CODE_RUNNER: Predict the output, run it, then fix the seeding bug so it prints the real warmest day.
public class PopcornMax {
    public static void main(String[] args) {
        int[] temps = {-12, -3, -18, -7};

        int max = 0;   // TODO: this seed is wrong. Predict what prints, then fix it.

        for (int i = 0; i < temps.length; i++) {
            if (temps[i] > max) {
                max = temps[i];
            }
        }

        System.out.println("Warmest temperature: " + max);
    }
}

PopcornMax.main(null);

### Homework Hack

**Task:** Write a complete Java program that reads a week of daily temperatures from an `int` array and prints a four-line weather report:

1.  The **average** temperature, as a decimal rather than a truncated integer.
2.  The **number of freezing days**, meaning days strictly below `32`.
3.  The **coldest temperature** and the **day index** it occurred on.
4.  Whether **every** day of the week was below freezing, as a `boolean`.

Then run your program on all three of these arrays and keep the printed output for each:

*   `{41, 28, 35, 19, 30, 52, 27}` — a mixed week
*   `{-12, -3, -18, -7, -22, -1, -9}` — entirely below zero, which is where a bad seed shows up
*   `{68, 71, 75, 70, 66, 80, 73}` — entirely above freezing, so the freezing count must be `0` and the all-freezing flag must be `false`

Use one loop per array, not four separate loops per array.

In [ ]:
// CODE_RUNNER: Print average, freezing-day count, coldest day with its index, and an all-below-freezing flag.
public class WeatherReport {
    public static void main(String[] args) {
        int[] temps = {41, 28, 35, 19, 30, 52, 27};
        int freezing = 32;

        // TODO 1: declare your four result variables HERE, before the loop.
        //         Seed the coldest value from temps[0], not from 0.

        for (int i = 0; i < temps.length; i++) {
            // TODO 2: add temps[i] to your running total
            // TODO 3: if this day is below freezing, add one to your count
            // TODO 4: if this day is colder than the coldest so far, update the value AND the index
            // TODO 5: if this day is NOT below freezing, turn your all-freezing flag off
        }

        // TODO 6: compute the average as a double, casting before you divide
        // TODO 7: print all four results, each on its own labeled line

    }
}

WeatherReport.main(null);

-----

## 4. Grading Plan (1 Point Total)

### Classroom Rubric

*   **0.2 points: Popcorn completion** — Student recorded a prediction, ran the cell, and corrected the seed so the printed maximum is `-3` rather than `0`.
*   **0.8 points: Homework completion**
    *   **0.4 correctness:** All four values are computed correctly on all three arrays, including the all-negative week. The coldest value and its index agree with each other.
    *   **0.3 completeness:** All three arrays were run and all four labeled lines are printed for each. The average prints as a decimal, not a truncated integer.
    *   **0.1 clarity:** Every result variable is declared before its loop, and the all-freezing flag starts at `true` and is only ever turned off.

### Quick Validation Checklist

*   Present: `// CODE_RUNNER:` as the first line and `ClassName.main(null);` as the last line of each submission cell.
*   Present: all four result variables declared above the `for` loop.
*   Absent: any accumulator declared or reset inside the loop body.
*   Absent: a minimum or maximum seeded with `0` instead of `temps[0]`.
*   Present: a cast to `double` before the division that produces the average.
*   Present: output from all three required arrays, with the all-negative week reporting a coldest value that appears in that array.
*   Present: freezing count of `0` and all-freezing flag of `false` for the entirely-above-freezing week.

-----

## 5. Lesson Revisions & Feedback Evidence

**Feedback Received:** During my peer practice run, my project team told me the original Homework Hack was four separate exercises wearing one hat. They had written four loops over the same array and said the repetition felt like busywork rather than practice. Separately, and more usefully, every single one of them predicted that a maximum seeded with `0` would still work on negative temperatures.

**Revision Made:** Two changes. First, I rewrote the Homework Hack to require all four values from a **single** pass over each array, which is both less typing and closer to how the free-response questions are actually phrased. Second, I moved the badly seeded maximum out of a footnote and made it the entire Popcorn Hack, structured as predict-then-run so the wrong answer prints before anyone fixes it. Watching `0` appear for a week of below-zero temperatures landed far harder than me explaining the rule ever did.

-----

## Resources

[AP Classroom](https://apstudents.collegeboard.org/courses/ap-computer-science-a)

[CED Selection and Iteration section](https://apcentral.collegeboard.org/media/pdf/ap-computer-science-a-course-and-exam-description.pdf)